In [0]:
# ============================================================
# Project  : ABC E-Commerce
# Layer    : Silver
# Notebook : 09_Silver_Payments
# Author   : Suriya
# Purpose  : Clean and Validate Payment Data
# ============================================================

from pyspark.sql.functions import *

# ------------------------------------------------------------
# Step 1 : Read Bronze Payments Table
# ------------------------------------------------------------

payments_df = spark.table("workspace.default.bronze_payments")

# ------------------------------------------------------------
# Step 2 : Verify Data
# ------------------------------------------------------------

payments_df.show(5)

payments_df.printSchema()

print("Total Records :", payments_df.count())

payments_df.describe().show()

# ------------------------------------------------------------
# Step 3 : Null Validation
# ------------------------------------------------------------

null_df = payments_df.filter(
    col("payment_id").isNull() |
    col("order_id").isNull() |
    col("payment_status").isNull() |
    col("payment_method").isNull()
)

print("Null Records :", null_df.count())

null_df.show()

# ------------------------------------------------------------
# Step 4 : Remove Null Records
# ------------------------------------------------------------

valid_df = payments_df.filter(
    col("payment_id").isNotNull() &
    col("order_id").isNotNull() &
    col("payment_status").isNotNull() &
    col("payment_method").isNotNull()
)

# ------------------------------------------------------------
# Step 5 : Duplicate Validation
# ------------------------------------------------------------

duplicate_df = (
    valid_df
    .groupBy("payment_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Records :", duplicate_df.count())

duplicate_df.show()

# ------------------------------------------------------------
# Step 6 : Remove Duplicate Records
# ------------------------------------------------------------

silver_df = valid_df.dropDuplicates(["payment_id"])

# ------------------------------------------------------------
# Step 7 : Business Rule Validation
# Payment Status should be SUCCESS or FAILED
# ------------------------------------------------------------

invalid_status_df = silver_df.filter(
    ~col("payment_status").isin("SUCCESS", "FAILED")
)

print("Invalid Status Records :", invalid_status_df.count())

invalid_status_df.show()

# ------------------------------------------------------------
# Step 8 : Remove Invalid Status
# ------------------------------------------------------------

silver_df = silver_df.filter(
    col("payment_status").isin("SUCCESS", "FAILED")
)

# ------------------------------------------------------------
# Step 9 : Add Audit Columns
# ------------------------------------------------------------

silver_df = (
    silver_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("data_source", lit("payments.csv"))
)

# ------------------------------------------------------------
# Step 10 : Verify Final Data
# ------------------------------------------------------------

silver_df.show(5)

silver_df.printSchema()

print("Silver Record Count :", silver_df.count())

# ------------------------------------------------------------
# Step 11 : Write Silver Table
# ------------------------------------------------------------

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_payments")

# ------------------------------------------------------------
# Step 12 : Verify Silver Table
# ------------------------------------------------------------

spark.sql("""
SELECT *
FROM workspace.default.silver_payments
LIMIT 10
""").show()